# Q-mini-light-0.7M — Personal AI Assistant (CPU-only)

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lion36508/Q-mini-light-0.7M/blob/main/colab/QMini_Personal_Assistant_CPU_One_Click.ipynb)

**इस notebook को सीधे Colab में खोलें — डाउनलोड करने की जरूरत नहीं।** ऊपर **Open in Google Colab** दबाएँ, फिर **Runtime → Run all** करें।

यह notebook repository के वास्तविक \`QMiniMoE\`, मौजूदा tokenizer और checkpoint से English-first Personal Assistant की CPU-only instruction fine-tuning करती है। TinyStories को इस SFT dataset में इस्तेमाल नहीं किया जाता।

- Dataset: 800 English/Hinglish conversational examples
- Existing chat tokens: \`<|system|>\`, \`<|user|>\`, \`<|assistant|>\`, \`<|end|>\`
- कोई नया vocabulary token या embedding resize नहीं
- प्रत्येक conversation एक अलग DataLoader sample
- Loss केवल Assistant response और end markers पर
- Physical batch 4 × gradient accumulation 2 = effective batch 8
- Context 512; CPU only; Google Drive checkpoints और Q2-v2 export

**महत्वपूर्ण:** यह notebook पहले Drive में trained Q-mini checkpoint और उसी का matching tokenizer खोजती है। Random step-zero \`initial_brain\` से SFT शुरू नहीं करेगी। अगर matching trained checkpoint नहीं मिलता, तो स्पष्ट error के साथ रुक जाएगी।

In [ ]:
# CPU-only guard — run before importing PyTorch.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CPU-only mode requested. CUDA will not be used.")

In [ ]:
# Clone the actual repository, install only missing lightweight dependencies, and mount Drive.
from pathlib import Path
import subprocess, sys, importlib.util, json, hashlib, random, time, shutil, math

REPO_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
REPO_DIR = Path("/content/Q-mini-light-0.7M")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)

missing = [name for name in ("tokenizers", "numpy") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

sys.path.insert(0, str(REPO_DIR))
import torch
from qmini.config import QMiniConfig
if torch.cuda.is_available():
    raise RuntimeError("CPU-only guard failed. Select Runtime → Change runtime type → CPU, then restart.")
torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DRIVE = DRIVE_ROOT / "QMini-Light-0.7M"
PROJECT_DRIVE.mkdir(parents=True, exist_ok=True)

repo_config = QMiniConfig.from_json(REPO_DIR / "configs/qmini_light_0.7m.json")
assert repo_config.estimated_parameter_count() == 705742
assert repo_config.max_context == 512
print("Repository:", REPO_URL)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Q-mini:", f"{repo_config.estimated_parameter_count():,} parameters | {repo_config.n_experts} experts | Top-{repo_config.top_k}")

In [ ]:
# Find a real trained checkpoint and its matching config/tokenizer in Google Drive.
from qmini.tokenizer import load_tokenizer

NUM_PAIRS = 800
OPTIMIZER_STEPS = 300
PHYSICAL_BATCH = 4
GRAD_ACCUM = 2
EFFECTIVE_BATCH = PHYSICAL_BATCH * GRAD_ACCUM
CONTEXT_LENGTH = 512
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.1
VALID_FRACTION = 0.10
SEED = 42
CPU_THREADS = 4
RUN_NAME = "assistant_sft_english_hinglish_800"

SEARCH_ROOTS = [
    PROJECT_DRIVE / "training_ready",
    PROJECT_DRIVE / "training_runs",
    PROJECT_DRIVE / "runs",
    PROJECT_DRIVE / "exports",
    PROJECT_DRIVE / "checkpoints",
]
excluded = {"initial_brain", "initial_brains", "assistant_sft_cpu_runs", "assistant_sft"}
candidates = []
for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file() or any(part.lower() in excluded for part in p.parts):
            continue
        if p.suffix.lower() == ".pt" or (p.suffix.lower() == ".npz" and "final_model" in [x.lower() for x in p.parts]):
            candidates.append(p)
candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)

BASE_MODEL_PATH = None
BASE_CONFIG_PATH = None
TOKENIZER_PATH = None
PACKAGE_DIR = None

def sha256_path(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for candidate in candidates:
    try:
        if candidate.suffix.lower() == ".pt":
            state = torch.load(candidate, map_location="cpu", weights_only=True)
            if not isinstance(state, dict):
                continue
            if state.get("format") == "qmini-checkpoint-v2":
                if int(state.get("step", 0)) <= 0 or state.get("config") is None:
                    continue
                cfg = QMiniConfig.from_dict(state["config"])
            elif "format" not in state and "model" not in state:
                cfg = repo_config
            else:
                continue
        else:
            import numpy as np
            with np.load(candidate, allow_pickle=False) as z:
                meta = json.loads(str(z["metadata_json"].item()))
            if meta.get("format") != "qmini-packed-v2":
                continue
            cfg = QMiniConfig.from_dict(meta["config"])

        if cfg.estimated_parameter_count() != 705742 or cfg.max_context != 512:
            continue

        package = None
        try:
            rel = candidate.relative_to((PROJECT_DRIVE / "training_ready").resolve())
            package = PROJECT_DRIVE / "training_ready" / rel.parts[0]
        except (ValueError, IndexError):
            pass

        config_options = []
        tokenizer_options = []
        if package:
            config_options += [package / "active_config.json", package / "initial_brain" / "config.json"]
            tokenizer_options += [package / "tokenizer" / "tokenizer.json"]
        config_options += [candidate.parent / "config.json", candidate.parent.parent / "config.json", REPO_DIR / "configs/qmini_light_0.7m.json"]
        tokenizer_options += [candidate.parent / "tokenizer.json", candidate.parent.parent / "tokenizer.json"]
        config_path = next((p for p in config_options if p.is_file()), None)
        tok_path = next((p for p in tokenizer_options if p.is_file()), None)
        if config_path is None or tok_path is None:
            continue
        test_cfg = QMiniConfig.from_json(config_path)
        test_tok = load_tokenizer(tok_path)
        if test_cfg.to_dict() != cfg.to_dict() or test_tok.get_vocab_size() != cfg.vocab_size:
            continue
        required = ("<pad>", "<bos>", "<eos>", "<|system|>", "<|user|>", "<|assistant|>", "<|end|>")
        if any(test_tok.token_to_id(t) is None for t in required):
            continue
        BASE_MODEL_PATH, BASE_CONFIG_PATH, TOKENIZER_PATH, PACKAGE_DIR = candidate, config_path, tok_path, package
        break
    except Exception as exc:
        print("Skipping incompatible checkpoint:", candidate, "|", type(exc).__name__, str(exc)[:180])

if BASE_MODEL_PATH is None:
    raise FileNotFoundError(
        "No compatible trained Q-mini checkpoint + matching tokenizer/config were found. "
        "Expected a trained latest.pt under MyDrive/QMini-Light-0.7M/training_ready/<package>/training_state/ "
        "or a Q2-v2 final_model/model.qmini.npz. The random initial_brain is intentionally rejected."
    )

CONFIG = QMiniConfig.from_json(BASE_CONFIG_PATH)
TOKENIZER = load_tokenizer(TOKENIZER_PATH)
assert CONFIG.max_context == CONTEXT_LENGTH and TOKENIZER.get_vocab_size() == CONFIG.vocab_size
print("SELECTED TRAINED MODEL:", BASE_MODEL_PATH)
print("CONFIG:", BASE_CONFIG_PATH)
print("TOKENIZER:", TOKENIZER_PATH)
print("Package:", PACKAGE_DIR)
print("Vocabulary preserved:", CONFIG.vocab_size, "| parameters:", f"{CONFIG.estimated_parameter_count():,}")
print("Effective batch:", EFFECTIVE_BATCH)

In [ ]:
# Generate 800 personal-assistant conversations (English + Hinglish); no story corpus.
SYSTEM_PROMPT = (
    "You are Q-mini, a friendly, practical and accurate personal AI assistant. "
    "Reply in English by default and use natural Hinglish when the user does. "
    "Be concise, helpful and honest. Do not tell stories unless explicitly asked."
)

SEEDS = {
"greeting": [
("Hi", "Hello sir, what are you doing? Welcome back."),
("Hello", "Hello sir! Good to see you. How can I help today?"),
("Hey", "Hey sir! I'm here and ready to help. What's on your mind?"),
("Good morning", "Good morning, sir! I hope your day is going well. How can I help?"),
("Good evening", "Good evening, sir! What would you like to work on today?"),
("How are you?", "I'm doing well, thank you. How can I help you today?"),
("What are you doing?", "I'm here and ready to help you, sir. What would you like to do?"),
("Are you there?", "Yes, sir. I'm here. Tell me what you need."),
("I'm back", "Welcome back, sir! What shall we work on next?"),
("Can we talk?", "Of course, sir. I'm listening. What would you like to talk about?")],
"general_qa": [
("What is a CPU?", "A CPU is the main general-purpose processor. It executes instructions and coordinates computer operations."),
("What is a GPU?", "A GPU handles many operations in parallel and is useful for graphics and machine-learning workloads."),
("What is RAM?", "RAM is fast temporary memory used by running programs. It is normally cleared when power is off."),
("What is Python?", "Python is a general-purpose programming language known for readable syntax and a large library ecosystem."),
("What is machine learning?", "Machine learning trains systems to recognize patterns in data and use them to make predictions."),
("What is a tokenizer?", "A tokenizer converts text into token IDs. The tokenizer vocabulary must match the model embeddings."),
("What is an epoch?", "An epoch is one complete pass through the training dataset."),
("What is overfitting?", "Overfitting happens when a model memorizes training examples and performs poorly on unseen examples."),
("What is a neural network?", "A neural network is a trainable system of mathematical operations that learns patterns from examples."),
("What is an AI assistant?", "An AI assistant helps with questions and tasks by interpreting user input and generating useful responses.")],
"personal_assistant": [
("Help me make a daily plan.", "List fixed commitments first, choose three important tasks, add breaks and leave a time buffer."),
("Help me organize my tasks.", "Write every task in one list, note deadlines and importance, then prioritize the urgent high-impact work."),
("Write a polite email asking for an update.", "Subject: Request for an Update\n\nHello,\n\nI hope you are doing well. Could you please share an update when convenient?\n\nThank you."),
("Make this more professional: send me the file now.", "Could you please send me the file at your earliest convenience?"),
("Summarize a document.", "Share the document text or upload it. I'll summarize the main points, details and action items."),
("Help me prepare for an interview.", "Review the role, prepare examples of your work and practice concise answers. I can run a mock interview."),
("Make a study timetable.", "Choose a daily study window, divide subjects into focused sessions and include short breaks and weekly revision."),
("Help me learn English.", "We can practice daily. I'll use simple English, correct mistakes gently and explain useful words with examples."),
("Remind me to take a break.", "A short break can help. Stand up, stretch, drink some water and return when you're ready."),
("I feel confused about a task.", "Tell me the goal and what is confusing. I'll divide the task into clear, manageable steps.")],
"technology_coding": [
("What is Git?", "Git is a version-control system that records project changes and helps developers collaborate."),
("What is GitHub?", "GitHub hosts Git repositories and provides tools for code review, issues and automation."),
("What is an API?", "An API is an interface that lets software systems exchange requests and responses."),
("What is JSON?", "JSON is a lightweight text format for storing and exchanging structured data."),
("What is a Python function?", "A function is a reusable block of code that performs a task and can accept inputs."),
("How do I debug an error?", "Read the full traceback, find the first relevant line in your code, reproduce the issue and test one fix at a time."),
("What is a pull request?", "A pull request proposes changes to a repository so they can be reviewed before merging."),
("What is a checkpoint?", "A checkpoint stores model weights and often optimizer and training state so training can resume."),
("What is CPU-only training?", "CPU-only training runs model operations on the processor without CUDA or a GPU."),
("Why should I test code?", "Tests catch regressions early and help verify that changes behave as intended.")],
"hinglish": [
("Kya kar rahe ho?", "I'm here, sir! Aap bataiye, main aapki kya help kar sakta hoon?"),
("Kaise ho?", "Main badhiya hoon, sir. Aap kaise hain?"),
("Mujhe English seekhni hai.", "Bilkul! Roz thodi practice karenge. Main simple English examples aur gentle corrections dunga."),
("Ye error kaise fix karun?", "Pehle poora error message aur traceback dekhiye. Aap error bhejiye, main step by step debug karne mein help karunga."),
("Mera code run nahi ho raha.", "Koi baat nahi. Error ka exact text aur related code bhejiye; hum ek-ek issue check karenge."),
("Mujhe simple words mein samjhao.", "Bilkul. Main simple language aur chhote examples ke saath samjhaunga."),
("Aaj kya karna chahiye?", "Aaj ke liye teen important tasks choose kijiye, ek ko pehle complete kijiye aur beech mein breaks lijiye."),
("Kya tum meri help karoge?", "Haan sir, zaroor. Aap bataiye kya kaam hai; main jitni ho sake utni practical help karunga."),
("Mujhe samajh nahi aa raha.", "Koi baat nahi. Jo part confusing hai woh bhejiye, main use aasaan steps mein explain karunga."),
("Training CPU par chal sakti hai?", "Haan, CPU par chal sakti hai, lekin GPU ke muqable slow hogi. Hum small batches aur checkpoints use karenge.")]
}

WRAPPERS = [
"{q}", "Please answer: {q}", "Can you help me with this? {q}", "I have a question: {q}",
"Explain simply: {q}", "Give me a clear answer: {q}", "Tell me briefly: {q}", "I want to understand: {q}",
"Please guide me: {q}", "Help me understand this: {q}", "Mujhe batao: {q}", "Iska jawab do: {q}",
"Please help karo: {q}", "Simple words mein samjhao: {q}", "Bhai, {q}", "Thoda guide karo: {q}"
]
records = []
for category, pairs in SEEDS.items():
    for seed_index, (question, answer) in enumerate(pairs):
        for wrapper in WRAPPERS:
            prompt = wrapper.format(q=question).strip()
            if category == "greeting" and question == "Hi" and wrapper == "{q}":
                prompt = "Hi"
            records.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": answer}
                ],
                "metadata": {
                    "category": category,
                    "language": "hinglish" if category == "hinglish" else "english",
                    "synthetic": True,
                    "seed_group": f"{category}:{seed_index}"
                }
            })
assert len(records) == 800
random.Random(SEED).shuffle(records)

OUTPUT_DIR = PROJECT_DRIVE / "assistant_sft_cpu_runs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = OUTPUT_DIR / "qmini_assistant_synthetic_800.jsonl"
if not DATASET_PATH.exists():
    with DATASET_PATH.open("w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
else:
    records = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]

assert len(records) == 800
assert any(m["content"] == "Hi" for r in records for m in r["messages"] if m["role"] == "user")
categories = {}
for row in records:
    c = row["metadata"]["category"]
    categories[c] = categories.get(c, 0) + 1
print("Dataset:", DATASET_PATH)
print("Conversation pairs:", len(records))
print("Category distribution:", categories)
print("Exact target:", "Hi → Hello sir, what are you doing? Welcome back.")

## Start training

नीचे वाले cell के बाईं ओर ▶️ दबाएँ। **Run all** करने पर यह cell अपने क्रम से चलेगा और CPU पर वास्तविक Q-mini SFT शुरू करेगा।

In [ ]:
# Build independent chat samples and begin CPU-only instruction fine-tuning.
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from qmini.model import QMiniMoE
from qmini.text import escape_special_token_literals
from qmini.packed_runtime import load_packed_state_into_trainable_model
from qmini.export import export_packed_state, verify_packed_export

# Reload the saved JSONL if the dataset-generation cell was skipped or the Colab runtime restarted.
if "records" not in globals():
    if "DATASET_PATH" not in globals():
        raise RuntimeError(
            "Dataset setup is missing. Run the cells above in order, especially the "
            "'Generate 800 personal-assistant conversations' cell, then rerun this training cell."
        )
    if not Path(DATASET_PATH).is_file():
        raise FileNotFoundError(
            f"Saved chat dataset was not found at {DATASET_PATH}. "
            "Run the dataset-generation cell above first."
        )
    with Path(DATASET_PATH).open("r", encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    print(f"Reloaded {len(records)} conversations from {DATASET_PATH}")

if len(records) != NUM_PAIRS:
    raise ValueError(
        f"Expected {NUM_PAIRS} chat records, found {len(records)}. "
        "Rerun the dataset-generation cell or choose a fresh output folder."
    )

assert PHYSICAL_BATCH == 4 and GRAD_ACCUM == 2 and EFFECTIVE_BATCH == 8
assert CONTEXT_LENGTH == CONFIG.max_context == 512
device = torch.device("cpu")
torch.manual_seed(SEED)
random.seed(SEED)

special = {t: TOKENIZER.token_to_id(t) for t in ("<pad>", "<bos>", "<eos>", "<|system|>", "<|user|>", "<|assistant|>", "<|end|>")}
if any(v is None for v in special.values()):
    raise ValueError("The matching existing tokenizer is missing required Q-mini chat tokens.")
special = {k: int(v) for k, v in special.items()}

groups = {}
for i, row in enumerate(records):
    group = row.get("metadata", {}).get("seed_group", f"record:{i}")
    groups.setdefault(group, []).append(i)
group_names = list(groups)
protected = "greeting:0"
split_names = [g for g in group_names if g != protected]
random.Random(SEED).shuffle(split_names)
n_valid = max(1, round(len(group_names) * VALID_FRACTION))
valid_groups = set(split_names[:n_valid])
valid_ids = {i for g in valid_groups for i in groups[g]}
train_records = [r for i, r in enumerate(records) if i not in valid_ids]
valid_records = [r for i, r in enumerate(records) if i in valid_ids]

class ChatDataset(Dataset):
    def __init__(self, rows):
        self.samples = []
        for row in rows:
            msg = {m["role"]: m["content"] for m in row["messages"] if m.get("role") in ("system", "user", "assistant")}
            system_ids = TOKENIZER.encode(escape_special_token_literals(msg.get("system", SYSTEM_PROMPT))).ids[:96]
            user_ids = TOKENIZER.encode(escape_special_token_literals(msg["user"])).ids
            assistant_ids = TOKENIZER.encode(escape_special_token_literals(msg["assistant"])).ids
            if not assistant_ids:
                continue
            budget = CONTEXT_LENGTH - 8
            overflow = max(0, len(system_ids) + len(user_ids) + len(assistant_ids) - budget)
            if overflow and len(user_ids) > 8:
                cut = min(overflow, len(user_ids) - 8)
                user_ids = user_ids[cut:]
                overflow -= cut
            if overflow and len(system_ids) > 8:
                cut = min(overflow, len(system_ids) - 8)
                system_ids = system_ids[:-cut]
                overflow -= cut
            if overflow:
                assistant_ids = assistant_ids[:max(1, len(assistant_ids) - overflow)]
            ids = [special["<bos>"]]
            mask = [0]
            ids += [special["<|system|>"]] + system_ids + [special["<|end|>"]]
            mask += [0] * (len(system_ids) + 2)
            ids += [special["<|user|>"]] + user_ids + [special["<|end|>"]]
            mask += [0] * (len(user_ids) + 2)
            ids += [special["<|assistant|>"]] + assistant_ids + [special["<|end|>"], special["<eos>"]]
            mask += [1] * (len(assistant_ids) + 3)
            if len(ids) > CONTEXT_LENGTH:
                raise ValueError("A chat sample exceeds 512 tokens after safe truncation.")
            self.samples.append((ids, mask))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        return self.samples[index]

def collate_chat(batch):
    max_len = max(len(ids) for ids, _ in batch)
    x = torch.full((len(batch), max_len - 1), special["<pad>"], dtype=torch.long)
    y = torch.full((len(batch), max_len - 1), -100, dtype=torch.long)
    for i, (ids_list, mask_list) in enumerate(batch):
        ids = torch.tensor(ids_list, dtype=torch.long)
        mask = torch.tensor(mask_list[1:], dtype=torch.bool)
        n = len(ids) - 1
        x[i, :n] = ids[:-1]
        targets = ids[1:]
        y[i, :n] = torch.where(mask, targets, torch.full_like(targets, -100))
    return {"input_ids": x, "labels": y}

train_data = ChatDataset(train_records)
valid_data = ChatDataset(valid_records)
if len(train_data) < 8 or not len(valid_data):
    raise ValueError("Train/validation split is too small.")
loader_gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_data, batch_size=PHYSICAL_BATCH, shuffle=True, drop_last=True, num_workers=0, collate_fn=collate_chat, generator=loader_gen)
valid_loader = DataLoader(valid_data, batch_size=PHYSICAL_BATCH, shuffle=False, drop_last=False, num_workers=0, collate_fn=collate_chat)

model = QMiniMoE(CONFIG).to(device)
if BASE_MODEL_PATH.suffix.lower() == ".npz":
    load_packed_state_into_trainable_model(model, BASE_MODEL_PATH)
else:
    base = torch.load(BASE_MODEL_PATH, map_location="cpu", weights_only=True)
    if base.get("format") == "qmini-checkpoint-v2":
        if base.get("config") != CONFIG.to_dict():
            raise ValueError("Pretraining checkpoint config does not match selected config.")
        model.load_state_dict(base["model"], strict=True)
    elif "format" not in base and "model" not in base:
        model.load_state_dict(base, strict=True)
    else:
        raise ValueError("Unsupported checkpoint format. Select a Q-mini checkpoint or Q2-v2 final model.")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
base_sha, data_sha, tok_sha = sha256_path(BASE_MODEL_PATH), sha256_path(DATASET_PATH), sha256_path(TOKENIZER_PATH)
checkpoint_path = OUTPUT_DIR / "assistant_sft_latest.pt"
metrics_path = OUTPUT_DIR / "assistant_sft_metrics.jsonl"
start_step = 0
if checkpoint_path.is_file():
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    expected = {"format": "qmini-chat-sft-v1", "base_sha": base_sha, "data_sha": data_sha, "tokenizer_sha": tok_sha, "config": CONFIG.to_dict()}
    for k, v in expected.items():
        if saved.get(k) != v:
            raise ValueError(f"Existing SFT checkpoint mismatch ({k}). Choose a new RUN_NAME instead of overwriting.")
    model.load_state_dict(saved["model"], strict=True)
    optimizer.load_state_dict(saved["optimizer"])
    start_step = int(saved["step"])
    print("Resuming existing SFT run at step", start_step)

def evaluate():
    model.eval()
    loss_sum, targets = 0.0, 0
    with torch.inference_mode():
        for bi, batch in enumerate(valid_loader):
            if bi >= 10:
                break
            out = model(batch["input_ids"], thinking_steps=CONFIG.thinking_steps)
            labels = batch["labels"]
            n = int((labels != -100).sum())
            if n:
                loss_sum += float(F.cross_entropy(out["logits"].float().reshape(-1, CONFIG.vocab_size), labels.reshape(-1), ignore_index=-100, reduction="sum"))
                targets += n
    model.train()
    if not targets:
        raise ValueError("Validation split has no assistant targets.")
    return loss_sum / targets

def save_checkpoint(step):
    payload = {
        "format": "qmini-chat-sft-v1", "step": step, "model": model.state_dict(),
        "optimizer": optimizer.state_dict(), "config": CONFIG.to_dict(),
        "base_sha": base_sha, "data_sha": data_sha, "tokenizer_sha": tok_sha,
        "torch_rng": torch.get_rng_state(), "loader_rng": loader_gen.get_state()
    }
    tmp = checkpoint_path.with_suffix(".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, checkpoint_path)

if start_step > 0 and checkpoint_path.is_file():
    torch.set_rng_state(saved["torch_rng"])
    loader_gen.set_state(saved["loader_rng"])
iterator = iter(train_loader)

model.train()
for step in range(start_step, OPTIMIZER_STEPS):
    optimizer.zero_grad(set_to_none=True)
    micro = []
    for _ in range(GRAD_ACCUM):
        try:
            micro.append(next(iterator))
        except StopIteration:
            iterator = iter(train_loader)
            micro.append(next(iterator))
    total_targets = sum(int((b["labels"] != -100).sum()) for b in micro)
    if total_targets <= 0:
        raise ValueError("No assistant targets in effective batch.")
    ce_sum_total = 0.0
    aux_total = 0.0
    for batch in micro:
        out = model(batch["input_ids"], thinking_steps=CONFIG.thinking_steps)
        ce_sum = F.cross_entropy(out["logits"].float().reshape(-1, CONFIG.vocab_size), batch["labels"].reshape(-1), ignore_index=-100, reduction="sum")
        ce = ce_sum / total_targets
        aux = out["aux_loss"].float()
        loss = ce + CONFIG.router_aux_loss_coef * aux / GRAD_ACCUM
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at step {step + 1}")
        loss.backward()
        ce_sum_total += float(ce_sum.detach())
        aux_total += float(aux.detach()) / GRAD_ACCUM
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    warmup = max(1, int(OPTIMIZER_STEPS * 0.03))
    if step < warmup:
        lr = LEARNING_RATE * (step + 1) / warmup
    else:
        prog = min(1.0, (step - warmup) / max(1, OPTIMIZER_STEPS - warmup))
        lr = LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))
    for group in optimizer.param_groups:
        group["lr"] = lr
    optimizer.step()
    completed = step + 1
    train_ce = ce_sum_total / total_targets
    valid_ce = evaluate() if completed % 25 == 0 or completed == OPTIMIZER_STEPS else None
    rec = {"step": completed, "train_assistant_ce": train_ce, "validation_assistant_ce": valid_ce, "assistant_target_tokens": total_targets, "router_aux_loss": aux_total, "gradient_norm": float(grad_norm), "learning_rate": lr, "device": "cpu"}
    with metrics_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    if completed % 5 == 0 or completed == 1 or completed == OPTIMIZER_STEPS:
        print(f"step {completed}/{OPTIMIZER_STEPS} | assistant CE={train_ce:.4f} | validation CE={valid_ce if valid_ce is not None else 'pending'}")
    if completed % 25 == 0 or completed == OPTIMIZER_STEPS:
        save_checkpoint(completed)

model.eval()
torch.save(model.state_dict(), OUTPUT_DIR / "assistant_sft_model.pt")
CONFIG.save(OUTPUT_DIR / "config.json")
shutil.copy2(TOKENIZER_PATH, OUTPUT_DIR / "tokenizer.json")
packed_path = OUTPUT_DIR / "model.qmini.npz"
export_packed_state(model.state_dict(), CONFIG, packed_path)
packed_report = verify_packed_export(packed_path)
assert packed_report.get("format") == "qmini-packed-v2"
manifest = {
    "task": "personal_assistant_instruction_sft",
    "device": "cpu",
    "parameters": sum(p.numel() for p in model.parameters()),
    "physical_batch": PHYSICAL_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch": EFFECTIVE_BATCH,
    "context_length": CONTEXT_LENGTH,
    "steps": OPTIMIZER_STEPS,
    "dataset_pairs": len(records),
    "train_pairs": len(train_data),
    "validation_pairs": len(valid_data),
    "base_model_sha256": base_sha,
    "dataset_sha256": data_sha,
    "tokenizer_sha256": tok_sha,
    "packed_format": "qmini-packed-v2"
}
(OUTPUT_DIR / "assistant_sft_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("\nSFT export complete:", packed_path)
print("Base weights:", BASE_MODEL_PATH)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# View the assistant training curve.
import pandas as pd
import matplotlib.pyplot as plt
metric_lines = [json.loads(x) for x in metrics_path.read_text(encoding="utf-8").splitlines() if x.strip()]
df = pd.DataFrame(metric_lines)
plt.figure(figsize=(10,4))
plt.plot(df["step"], df["train_assistant_ce"], label="Training assistant CE")
valid = df.dropna(subset=["validation_assistant_ce"])
if len(valid):
    plt.plot(valid["step"], valid["validation_assistant_ce"], marker="o", label="Validation assistant CE")
plt.xlabel("Optimizer step")
plt.ylabel("Cross-entropy loss")
plt.title("Q-mini Personal Assistant — CPU SFT")
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()
display(df.tail(10))

In [ ]:
# Load the exported Q2-v2 model on CPU and test the greeting + Hinglish.
from qmini.generate import load_model, generate
assistant_model, assistant_tokenizer, loaded_device = load_model(str(OUTPUT_DIR), device="cpu")
assistant_model.eval()
SYSTEM = "You are Q-mini, a friendly and practical personal assistant. Reply in English by default and natural Hinglish when appropriate."
def ask_qmini(text):
    prompt = f"<|system|>{SYSTEM}\n<|end|>\n<|user|>{text}\n<|end|>\n<|assistant|>"
    return generate(assistant_model, assistant_tokenizer, prompt, max_new_tokens=64, temperature=0.25, top_k=20, top_p=0.9, thinking_steps=assistant_model.config.thinking_steps, serialized_prompt=True)
print("Loaded device:", loaded_device)
print("USER: Hi")
print("Q-MINI:", ask_qmini("Hi"))
print()
print("USER: Kya kar rahe ho?")
print("Q-MINI:", ask_qmini("Kya kar rahe ho?"))
print("\nSaved files:", [p.name for p in OUTPUT_DIR.iterdir() if p.is_file()])